In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma


loader=TextLoader("rag_query_enhancement_dataset_211_lines_no_numbers.txt")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vstore=Chroma.from_documents(chunks,embedding_model)

retriever=vstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

C:\Users\Admin\AppData\Local\Temp\ipykernel_1164\32595226.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000025F63644910>, search_type='mmr', search_kwargs={'k': 5})

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(
    model="qwen/qwen3-vl-4b",
    base_url="http://127.0.0.1:1234/v1",
    api_key="dummy",
    temperature=0.7
    

)


decompose_prompt=ChatPromptTemplate.from_template("""
Break the question into 3 smaller questions.

Question: {question}

Return only the question , one per line 
""")

decompose_chain=decompose_prompt |  llm | StrOutputParser()

rag_prompt=ChatPromptTemplate.from_template("""
Answer the question using the context.

Question:
{question}

Context:
{context}

""")

rag_chain=rag_prompt | llm | StrOutputParser()

question="""
What is RAG
Explain their advantage and when to use each.

"""

sub_questions=decompose_chain.invoke({
    "question":question
}).split("\n")

docs=[]

sub_questions = [
    q.strip("0123456789.- ")
    for q in decompose_chain.invoke({
        "question": question
    }).split("\n")
    if q.strip()
]

for q in sub_questions:
    docs.extend(retriever.invoke(q))

context="\n\n".join(
    doc.page_content
    for doc in docs
)

answer=rag_chain.invoke({
    'question':question,
    'context':context
})

print(answer)

**What is RAG?**

**RAG** stands for **Retrieval Augmented Generation**. It is a system architecture that combines two key components:

1. **Retrieval**: A search mechanism that fetches relevant external information (e.g., documents, manuals, websites, databases, or internal knowledge) based on the user’s query.
2. **Generation**: A language model that generates an answer based on the retrieved context.

In RAG, the system first retrieves relevant external knowledge before asking the language model to generate an answer. This approach ensures that the model answers are grounded in current or domain-specific information, rather than relying solely on its internal parameters.

---

**Advantages of RAG:**

1. **Reduces Dependence on Model Parameters**: RAG leverages external knowledge sources, meaning the model doesn’t need to be retrained to answer questions accurately — it can be fine-tuned or updated with minimal effort.
2. **Provides Current Information**: RAG can incorporate up-to-da